# Phase 2 — Kaggle dataset acquisition

This notebook runs against a Kaggle Jupyter Server from VS Code. It acquires and validates raw summarization datasets without downloading them to the local laptop.

**Phase boundary:** acquisition and raw-data validation only. Cleaning, normalization, deduplication, token analysis, and schema conversion begin in Phase 3.

In [ ]:
from __future__ import annotations

import os
import platform
import shutil
import sys
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")

if not KAGGLE_WORKING.is_dir():
    raise RuntimeError(
        "This notebook is not connected to a Kaggle runtime. "
        "Select the Kaggle Jupyter Server kernel before continuing."
    )

PROJECT_ROOT = KAGGLE_WORKING / "text-summarizer"
OUTPUT_ROOT = PROJECT_ROOT / "phase2"
SCRATCH_ROOT = Path("/kaggle/temp/text-summarizer")
HF_HOME = SCRATCH_ROOT / "huggingface"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

disk = shutil.disk_usage(KAGGLE_WORKING)
print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Executable: {sys.executable}")
print(f"Kaggle input: {KAGGLE_INPUT}")
print(f"Kaggle working: {KAGGLE_WORKING}")
print(f"Free working disk: {disk.free / (1024**3):.2f} GiB")
print("Remote Kaggle runtime validation passed.")

In [ ]:
from importlib.metadata import PackageNotFoundError, version

REQUIRED_PACKAGES = ("datasets", "huggingface_hub", "pandas", "pyarrow")
missing_packages: list[str] = []

for package_name in REQUIRED_PACKAGES:
    try:
        print(f"{package_name}: {version(package_name)}")
    except PackageNotFoundError:
        missing_packages.append(package_name)
        print(f"{package_name}: missing")

if missing_packages:
    print("Missing packages will be installed only after this environment report is reviewed.")
else:
    print("Dataset dependencies are available.")

## Acquisition helpers

The next cell records split sizes, required columns, fingerprints, and first-example character lengths. It does not clean, transform, or resplit any dataset.

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from typing import Any

from datasets import DatasetDict, load_dataset

MANIFEST_PATH = OUTPUT_ROOT / "dataset_manifest.json"
dataset_manifest: dict[str, Any] = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "datasets_package_version": version("datasets"),
    "datasets": {},
}

def record_hugging_face_dataset(
    *,
    key: str,
    repository: str,
    dataset: DatasetDict,
    source_column: str,
    summary_column: str,
    config: str | None = None,
) -> None:
    record: dict[str, Any] = {
        "kind": "huggingface",
        "repository": repository,
        "config": config,
        "source_column": source_column,
        "summary_column": summary_column,
        "splits": {},
    }

    print(f"\n{key}: {repository}")
    for split_name, split_dataset in dataset.items():
        required = {source_column, summary_column}
        missing = required.difference(split_dataset.column_names)
        if missing:
            raise ValueError(f"{key}/{split_name} is missing columns: {sorted(missing)}")

        example = split_dataset[0] if split_dataset.num_rows else {}
        source_length = len(str(example.get(source_column, "")))
        summary_length = len(str(example.get(summary_column, "")))
        record["splits"][split_name] = {
            "rows": split_dataset.num_rows,
            "columns": list(split_dataset.column_names),
            "fingerprint": split_dataset._fingerprint,
            "first_source_characters": source_length,
            "first_summary_characters": summary_length,
        }
        print(
            f"  {split_name}: {split_dataset.num_rows:,} rows; "
            f"first source/summary chars={source_length:,}/{summary_length:,}"
        )

    dataset_manifest["datasets"][key] = record

def clone_repository(*, key: str, url: str, destination: Path) -> Path:
    git_directory = destination / ".git"
    if destination.exists() and not git_directory.is_dir():
        raise RuntimeError(f"Refusing to overwrite non-Git path: {destination}")
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", url, str(destination)],
            check=True,
        )

    commit = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"],
        text=True,
    ).strip()
    dataset_manifest["datasets"][key] = {
        "kind": "git",
        "repository": url,
        "commit": commit,
        "path": str(destination),
    }
    print(f"{key}: {commit}")
    return destination

def save_manifest() -> None:
    MANIFEST_PATH.write_text(
        json.dumps(dataset_manifest, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    print(f"Manifest saved: {MANIFEST_PATH}")

print("Acquisition helpers are ready.")

## 1. CNN/DailyMail — general articles

Downloads the official Hugging Face `3.0.0` configuration and validates its existing splits.

In [ ]:
cnn_dailymail = load_dataset("abisee/cnn_dailymail", "3.0.0")
record_hugging_face_dataset(
    key="cnn_dailymail",
    repository="abisee/cnn_dailymail",
    config="3.0.0",
    dataset=cnn_dailymail,
    source_column="article",
    summary_column="highlights",
)
save_manifest()

## 2. GovReport — long reports

Downloads long government reports with multi-paragraph reference summaries.

In [ ]:
govreport = load_dataset("ccdv/govreport-summarization")

record_hugging_face_dataset(
    key="govreport",
    repository="ccdv/govreport-summarization",
    dataset=govreport,
    source_column="report",
    summary_column="summary",
)

validation_first = govreport["validation"][0]
test_first = govreport["test"][0]

same_source = validation_first["report"] == test_first["report"]
same_summary = validation_first["summary"] == test_first["summary"]

print(f"Validation/test first source identical: {same_source}")
print(f"Validation/test first summary identical: {same_summary}")

if same_source or same_summary:
    raise ValueError(
        "GovReport validation and test first records are not independent."
    )

save_manifest()

## 3. BillSum — legal documents

Downloads legislative text paired with human-written bill summaries.

In [ ]:
billsum = load_dataset("FiscalNote/billsum")

record_hugging_face_dataset(
    key="billsum",
    repository="FiscalNote/billsum",
    dataset=billsum,
    source_column="text",
    summary_column="summary",
)

save_manifest()

## 4. QMSum — multi-speaker meetings

Clones the official MIT-licensed repository into temporary Kaggle storage and records its exact Git commit.

In [ ]:
qmsum_root = clone_repository(
    key="qmsum",
    url="https://github.com/Yale-LILY/QMSum.git",
    destination=SCRATCH_ROOT / "qmsum",
)

qmsum_data_root = qmsum_root / "data" / "ALL"

qmsum_files = sorted(
    path
    for path in qmsum_data_root.rglob("*")
    if path.is_file()
)

if not qmsum_files:
    raise RuntimeError(
        f"No QMSum data files found under {qmsum_data_root}"
    )

dataset_manifest["datasets"]["qmsum"]["data_files"] = [
    str(path.relative_to(qmsum_root))
    for path in qmsum_files
]

print(f"QMSum data files: {len(qmsum_files)}")

for path in qmsum_files[:10]:
    print(f"  {path.relative_to(qmsum_root)}")

save_manifest()

## 5. TweetSumm — customer support

Clones the official annotation repository and checks whether Kaggle's Customer Support on Twitter input containing `twcs.csv` is attached.

In [ ]:
import kagglehub

tweetsumm_root = clone_repository(
    key="tweetsumm",
    url="https://github.com/guyfe/Tweetsumm.git",
    destination=SCRATCH_ROOT / "tweetsumm",
)

tweetsumm_annotation_files = sorted(
    tweetsumm_root.rglob("*.jsonl")
)

if not tweetsumm_annotation_files:
    raise RuntimeError(
        "TweetSumm JSONL annotation files were not found."
    )

# Prefer an attached Kaggle input when available.
mounted_twcs_candidates = sorted(
    KAGGLE_INPUT.rglob("twcs.csv")
)

twcs_candidates = mounted_twcs_candidates
twcs_acquisition = "kaggle_input"

# Otherwise download into Kaggle temporary storage.
if not twcs_candidates:
    twcs_download_root = Path(
        kagglehub.dataset_download(
            "thoughtvector/customer-support-on-twitter",
            output_dir=str(
                SCRATCH_ROOT / "customer-support-on-twitter"
            ),
        )
    )

    twcs_candidates = sorted(
        twcs_download_root.rglob("twcs.csv")
    )
    twcs_acquisition = "kagglehub"

if not twcs_candidates:
    raise RuntimeError(
        "Customer Support on Twitter twcs.csv was not found."
    )

dataset_manifest["datasets"]["tweetsumm"].update(
    {
        "annotation_files": [
            str(path.relative_to(tweetsumm_root))
            for path in tweetsumm_annotation_files
        ],
        "customer_support_input_attached": bool(
            mounted_twcs_candidates
        ),
        "customer_support_input_available": bool(
            twcs_candidates
        ),
        "customer_support_acquisition": twcs_acquisition,
        "customer_support_input_paths": [
            str(path) for path in twcs_candidates
        ],
    }
)

print(
    f"TweetSumm annotation files: "
    f"{len(tweetsumm_annotation_files)}"
)
print(
    f"Customer Support on Twitter files: "
    f"{len(twcs_candidates)}"
)
print(
    f"Customer Support acquisition: "
    f"{twcs_acquisition}"
)

save_manifest()

## Phase 2 acquisition gate

Phase 2 acquisition is complete only after all five sources are accessible, their expected fields or files are present, and `dataset_manifest.json` records their versions. No preprocessing is performed in this notebook section.

In [ ]:
expected_datasets = {
    "cnn_dailymail",
    "govreport",
    "billsum",
    "qmsum",
    "tweetsumm",
}

recorded_datasets = set(
    dataset_manifest.get("datasets", {})
)

missing_datasets = expected_datasets.difference(
    recorded_datasets
)

if missing_datasets:
    raise RuntimeError(
        f"Missing dataset records: {sorted(missing_datasets)}"
    )

required_splits = {
    "cnn_dailymail": {"train", "validation", "test"},
    "govreport": {"train", "validation", "test"},
    "billsum": {"train", "test", "ca_test"},
}

for dataset_key, expected_splits in required_splits.items():
    recorded_splits = set(
        dataset_manifest["datasets"][dataset_key]
        .get("splits", {})
    )

    missing_splits = expected_splits.difference(
        recorded_splits
    )

    if missing_splits:
        raise RuntimeError(
            f"{dataset_key} is missing splits: "
            f"{sorted(missing_splits)}"
        )

qmsum_record = dataset_manifest["datasets"]["qmsum"]

if (
    not qmsum_record.get("commit")
    or not qmsum_record.get("data_files")
):
    raise RuntimeError(
        "QMSum commit or data-file inventory is missing."
    )

tweetsumm_record = dataset_manifest["datasets"]["tweetsumm"]

if not tweetsumm_record.get("commit"):
    raise RuntimeError("TweetSumm commit is missing.")

if not tweetsumm_record.get("annotation_files"):
    raise RuntimeError(
        "TweetSumm annotation inventory is missing."
    )

if not tweetsumm_record.get(
    "customer_support_input_available"
):
    raise RuntimeError(
        "TweetSumm customer-support source is unavailable."
    )

if not tweetsumm_record.get(
    "customer_support_input_paths"
):
    raise RuntimeError(
        "TweetSumm customer-support path is missing."
    )

save_manifest()

if not MANIFEST_PATH.is_file():
    raise RuntimeError(
        f"Manifest was not written: {MANIFEST_PATH}"
    )

print(f"Recorded datasets: {sorted(recorded_datasets)}")
print(f"Manifest: {MANIFEST_PATH}")
print(
    f"Manifest size: "
    f"{MANIFEST_PATH.stat().st_size:,} bytes"
)
print("Phase 2 raw-data acquisition gate passed.")